In [264]:
using Healpix
using LinearAlgebra
using StaticArrays
using Base.Threads
using WignerD
using NPZ
using Plots
using Falcons
#using PyPlot
#using PyCall
using BenchmarkTools
using SparseArrays

#hp = pyimport("healpy")
#include("../src/EBC.jl")

In [265]:
mutable struct ConvolutionSky{I<:Int, MC<:Matrix{Complex{Float64}}}
    lmax::I
    alm::MC
    realization::I
end

mutable struct ConvolutionBeam{I<:Int, MC<:Matrix{Complex{Float64}}}
    lmax::I
    mmax::I
    blm::MC
end

#=
mutable struct ConvolutionCalculate{I<:Int, Bl<:Bool}
    nside_output::I
    lmax_calculate::I
    mmax_calculate::I
    HWP::Bl
end
=#

mutable struct ConvolutionCalculate{I<:Int, Bl<:Bool}
    nside_output::I
    lstart::I
    lstop::I
    mmax_calculate::I
    HWP::Bl
end



In [266]:
function ConvolutionSky(;
        lmax::Int = 3*128 - 1,
        alm::Matrix{ComplexF64} = fill(1.0 + 1.0im, 2, 2),
        realization::Int = 1
    )
    return ConvolutionSky(
        lmax,
        alm,
        realization
    )
end

function ConvolutionBeam(;
    lmax::Int = 3*128-1,
    mmax::Int = 2,
    blm::Matrix{ComplexF64} = fill(1.0 + 1.0im, 2, 2)
    )
    return ConvolutionBeam(lmax, mmax, blm)
end

function ConvolutionCalculate(;
    nside_output::Int = 128,
    lstart::Int = 0,
    lstop::Int = 3*nside_output - 1,
    mmax_calculate::Int = 2,
    HWP::Bool = false
)
    lstart <= lstop || throw(ArgumentError("lstart must be <= lstop (got lstart=$lstart, lstop=$lstop)"))
    mmax_calculate <= lstop || throw(ArgumentError("mmax_calculate must be <= lstop (got mmax_calculate=$mmax_calculate, lstop=$lstop)"))
    return ConvolutionCalculate(
        nside_output,
        lstart,
        lstop,
        mmax_calculate,
        HWP
    )
end


ConvolutionCalculate

In [267]:
#methods(ConvolutionSky)

In [448]:
nside_in = 32
lmax_in = 3nside_in-1
fwhm = deg2rad(1.0)

0.017453292519943295

In [449]:
cs = ConvolutionSky()
fieldnames(typeof(cs))

(:lmax, :alm, :realization)

In [450]:
cb = ConvolutionBeam()
fieldnames(typeof(cb))

(:lmax, :mmax, :blm)

In [451]:
cc = ConvolutionCalculate(nside_output = nside_in, lstart = 0, mmax_calculate = 2)
fieldnames(typeof(cc))

(:nside_output, :lstart, :lstop, :mmax_calculate, :HWP)

In [457]:
include("../../BeamToyModel/src/BeamToyModel.jl")

Main.BeamToyModel

In [458]:
blm_harmonic = BeamToyModel.gaussian_harmonic(lmax_in, fwhm, mmax = lmax_in)

(blmT = Alm{ComplexF64, Vector{ComplexF64}}(ComplexF64[0.28209479177387814 + 0.0im, 0.4885756718694027 + 0.0im, 0.6306791852123736 + 0.0im, 0.7461067059896269 + 0.0im, 0.8458196072043231 + 0.0im, 0.9348319547260208 + 0.0im, 1.0159345688951988 + 0.0im, 1.0908692242925033 + 0.0im, 1.1608087185364095 + 0.0im, 1.226586793157029 + 0.0im  …  0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im], 95, 95, 191), blmE = Alm{ComplexF64, Vector{ComplexF64}}(ComplexF64[0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im  …  0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im], 95, 95, 191), blmB = Alm{ComplexF64, Vector{ComplexF64}}(ComplexF64[0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im  …

In [459]:
cb.lmax = lmax_in
cb.mmax = 2
cb.blm = hcat(blm_harmonic.blmT.alm,blm_harmonic.blmE.alm,blm_harmonic.blmB.alm) 
;

In [460]:
nalm = Healpix.numberOfAlms(lmax_in, lmax_in)
almT = rand(ComplexF64, nalm)
almE = rand(ComplexF64, nalm)
almB = rand(ComplexF64, nalm)
cs.lmax = lmax_in
cs.alm = hcat(almT, almE, almB)
;

In [461]:
#=
function lrange_idx(l::Int, lstart::Int)
    l >= lstart || throw(ArgumentError("l must be >= lstart"))
    offset = sum(2k + 1 for k in lstart:l-1; init=0)
    start = offset + 1
    stop  = start + (2l + 1) - 1
    return start, stop
end
=#
#=
function lmr_idx(;l::Int, m::Int, lstart::Int)
    l >= lstart || throw(ArgumentError("l must be >= lstart"))
    (-l <= m <= l) || throw(ArgumentError("m must be in [-l, l]"))
    offset = sum(2k + 1 for k in lstart:l-1; init=0)
    idx = offset + (m + l) + 1   # m=-l -> +1, m=l -> +2l+1
    return idx
end
=#
function lmr_idx(; l::Int, m::Int, lstart::Int, mmax::Int)
    l ≥ lstart || throw(ArgumentError("l must be >= lstart"))
    mcap = min(l, mmax)
    (-mcap ≤ m ≤ mcap) || throw(ArgumentError("m must be in [-min(l,mmax), min(l,mmax)]"))
    offset = 0
    for k in lstart:l-1
        offset += 2*min(k, mmax) + 1
    end
    return offset + (m + mcap) + 1
end
#=
function alm_idx(;l::Integer, m::Integer, lmax::Integer)
    return Int(m * (2 * lmax + 1 - m) // 2 + l)+1
end
=#
function alm_idx(; l::Integer, m::Integer, lmax::Integer, mmax::Integer=lmax)
    (0 ≤ m ≤ mmax) || throw(ArgumentError("m must be in [0, mmax]"))
    (m ≤ l ≤ lmax) || throw(ArgumentError("l must satisfy m ≤ l ≤ lmax"))

    # offset for all previous m-blocks
    offset = m * (lmax + 1) - (m * (m - 1)) ÷ 2
    return Int(offset + (l - m) + 1)  # 1-based
end

function blm_lrange(cb, cc)
    n = sum(2*min(l, cc.mmax_calculate) + 1 for l in cc.lstart:cc.lstop)
    blm_calc = Matrix{ComplexF64}(undef, n, 3)
    for l in cc.lstart:cc.lstop
        m = 0
        idx_in = alm_idx(l=l, m=m, lmax=cb.lmax, mmax = cb.mmax)
        idx_out= lmr_idx(l=l, m=m, lstart=cc.lstart, mmax=cc.mmax_calculate)
        blm_calc[idx_out,1] = cb.blm[idx_in,1]
        blm_calc[idx_out,2] = -(cb.blm[idx_in,2] + 1im*cb.blm[idx_in,3]) # spin2 = -(E +iB)
        blm_calc[idx_out,3] = -(cb.blm[idx_in,2] - 1im*cb.blm[idx_in,3]) # spin-2 = -(E -iB)
        for m in 1:min(l, cc.mmax_calculate)
            phase = isodd(m) ? -1.0 : 1.0
            idx_in = alm_idx(l=l, m=m, lmax=cb.lmax, mmax = cb.mmax)
            idx_out_positive= lmr_idx(l=l, m=m, lstart=cc.lstart, mmax=cc.mmax_calculate)
            idx_out_negative= lmr_idx(l=l, m=-m, lstart=cc.lstart, mmax=cc.mmax_calculate)
            # m positive
            blm_calc[idx_out_positive,1] = cb.blm[idx_in,1]
            blm_calc[idx_out_positive,2] = -(cb.blm[idx_in,2] + 1im*cb.blm[idx_in,3]) # spin2 = -(E +iB)
            blm_calc[idx_out_positive,3] = -(cb.blm[idx_in,2] - 1im*cb.blm[idx_in,3]) # spin-2 = -(E -iB)
            # m negative
            blm_calc[idx_out_negative,1] = conj(cb.blm[idx_in,1])*phase # conj(spin0)*(-1)^m 
            blm_calc[idx_out_negative,2] = conj(blm_calc[idx_out_positive,3])*phase # conj(spin-2)*(-1)^m 
            blm_calc[idx_out_negative,3] = conj(blm_calc[idx_out_positive,2])*phase # conj(spin2)*(-1)^m 
        end
    end
    return blm_calc
end

blm_lrange (generic function with 1 method)

In [462]:
test = blm_lrange(cb, cc)

474×3 Matrix{ComplexF64}:
 0.282095+0.0im      -0.0-0.0im     -0.0-0.0im
     -0.0+0.0im       0.0-0.0im      0.0-0.0im
 0.488576+0.0im      -0.0-0.0im     -0.0-0.0im
      0.0+0.0im      -0.0-0.0im     -0.0-0.0im
      0.0-0.0im   0.63061+0.0im     -0.0+0.0im
     -0.0+0.0im       0.0-0.0im      0.0-0.0im
 0.630679+0.0im      -0.0-0.0im     -0.0-0.0im
      0.0+0.0im      -0.0-0.0im     -0.0-0.0im
      0.0+0.0im      -0.0-0.0im  0.63061-0.0im
      0.0-0.0im  0.746025+0.0im     -0.0+0.0im
     -0.0+0.0im       0.0-0.0im      0.0-0.0im
 0.746107+0.0im      -0.0-0.0im     -0.0-0.0im
      0.0+0.0im      -0.0-0.0im     -0.0-0.0im
         ⋮                       
      0.0+0.0im      -0.0-0.0im     -0.0-0.0im
      0.0+0.0im      -0.0-0.0im   3.0338-0.0im
      0.0-0.0im   3.03427+0.0im     -0.0+0.0im
     -0.0+0.0im       0.0-0.0im      0.0-0.0im
  3.03461+0.0im      -0.0-0.0im     -0.0-0.0im
      0.0+0.0im      -0.0-0.0im     -0.0-0.0im
      0.0+0.0im      -0.0-0.0im  3.03427-0.0im


In [463]:
function alm_lrange(cs, cc)
    n = sum(2*l + 1 for l in cc.lstart:cc.lstop)
    alm_calc = Matrix{ComplexF64}(undef, n, 3)
    for l in cc.lstart:cc.lstop
        m = 0
        idx_in = alm_idx(l=l, m=m, lmax=cs.lmax)
        idx_out= lmr_idx(l=l, m=m, lstart=cc.lstart, mmax=cs.lmax)
        alm_calc[idx_out,1] = cs.alm[idx_in,1]
        alm_calc[idx_out,2] = cs.alm[idx_in,2]
        alm_calc[idx_out,3] = cs.alm[idx_in,3]
        for m in 1:l
            phase = isodd(m) ? -1.0 : 1.0
            idx_in = alm_idx(l=l, m=m, lmax=cs.lmax)
            idx_out_positive= lmr_idx(l=l, m=m, lstart=cc.lstart, mmax=cs.lmax)
            idx_out_negative= lmr_idx(l=l, m=-m, lstart=cc.lstart, mmax=cs.lmax)
            # m positive
            alm_calc[idx_out_positive,1] = cs.alm[idx_in,1]                           # spin0
            alm_calc[idx_out_positive,2] = -(cs.alm[idx_in,2] + 1im*cs.alm[idx_in,3]) # spin2 = -(E +iB)
            alm_calc[idx_out_positive,3] = -(cs.alm[idx_in,2] - 1im*cs.alm[idx_in,3]) # spin2 = -(E +iB)
            # m negative
            alm_calc[idx_out_negative,1] = conj(cs.alm[idx_in,1])*phase             # conj(spin0)
            alm_calc[idx_out_negative,2] = conj(alm_calc[idx_out_positive,3])*phase # conj(spin2)*(-1)^m
            alm_calc[idx_out_negative,3] = conj(alm_calc[idx_out_positive,2])*phase # conj(spin-2)*(-1)^m
        end
    end
    return alm_calc
end

alm_lrange (generic function with 1 method)

In [464]:
alm_lrange(cs, cc)

9216×3 Matrix{ComplexF64}:
   0.161017+0.0371979im   0.972445+0.571507im    0.10252+0.725487im
  -0.365392+0.143846im    0.666967-0.176962im   0.598605-0.33651im
   0.110148+0.208309im    0.533363+0.143328im   0.620734+0.323061im
   0.365392+0.143846im   -0.598605-0.33651im   -0.666967-0.176962im
  0.0497512-0.920093im     -1.1766-0.314737im   0.211441+1.26588im
  -0.421691+0.576638im     1.24337-0.239922im   0.730369-0.789138im
   0.394422+0.440436im    0.145565+0.443087im   0.558453+0.61594im
   0.421691+0.576638im   -0.730369-0.789138im   -1.24337-0.239922im
  0.0497512+0.920093im    0.211441-1.26588im     -1.1766+0.314737im
  -0.332009+0.043689im     1.33586-0.512514im  -0.388652-0.623598im
   0.528503-0.298124im     -1.2117-0.514258im  -0.282857+1.02769im
  -0.462039+0.528902im     0.48261-0.011566im   0.375762-0.492519im
  0.0561396+0.454624im    0.753393+0.42981im    0.151904+0.641035im
           ⋮                                   
   0.220153+0.959413im   -0.259117-0.869193im

## wignerD

In [465]:
n_beam = sum(2*min(l, cb.mmax) + 1 for l in cc.lstart:cc.lstop)
D_beam = spzeros(n_beam, n_beam)

n_sky = sum(2*l + 1 for l in cc.lstart:cc.lstop)
D_sky = spzeros(n_sky, n_beam)

9216×474 SparseMatrixCSC{Float64, Int64} with 0 stored entries:
⎡⠀⠀⠀⎤
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎣⠀⠀⠀⎦

In [466]:
include("../src/function2/rotator.jl")

fill_wignerD_band_block_avg! (generic function with 1 method)

In [467]:
ell = 100

100

In [468]:
θ = pi/3
φ = pi/4

0.7853981633974483

In [469]:
nptg = 1
θ = pi/3
φ = pi/4
dθ = rand(nptg)*1e-2
dφ = rand(nptg)*1e-2
ψ = rand(nptg)*2pi
;

In [470]:
alphas = zeros(size(dθ))
betas = zeros(size(dθ))
gammas = zeros(size(dθ))

for i in eachindex(dθ)
    err, (alphas[i], betas[i], gammas[i]) = check_split(φ, θ, dφ[i], dθ[i], ψ[i])
    if err >= 1e0
        @show err
    end
end

In [471]:
# first
# 1step version
function test1(ell, φ, θ, ψ)
    D_result = zeros(ComplexF64, 2ell+1, 2ell+1)
    for i in eachindex(φ)
        D_result .+= WignerD.wignerD(ell, φ[i], θ[i], ψ[i])
    end
    return D_result./length(φ)
end

test1 (generic function with 1 method)

In [472]:
D1 = test1(ell, φ.+dφ, θ.+dθ, ψ)

201×201 Matrix{ComplexF64}:
  -1.4729e-13+7.95561e-14im  …    1.1649e-15+2.15487e-15im
  4.28099e-13-1.44906e-12im      1.51336e-15+4.46514e-16im
  4.23071e-12+7.72522e-12im      1.47895e-15-8.10628e-16im
 -4.01566e-11-1.16108e-11im     -5.94172e-16+2.05771e-15im
  1.49731e-10-8.31322e-11im     -2.08144e-15-3.74579e-15im
 -1.70382e-10+6.02346e-10im  …   3.37254e-15+9.52681e-16im
  -1.02183e-9-1.81553e-9im      -4.72844e-15+2.66348e-15im
   6.17105e-9+1.70697e-9im       7.62668e-16-2.76099e-15im
  -1.59469e-8+9.0975e-9im        2.30625e-15+4.03929e-15im
   1.29209e-8-4.77887e-8im      -1.82452e-15-4.92615e-16im
   6.32219e-8+1.09343e-7im   …   2.95304e-15-1.70884e-15im
  -2.96285e-7-7.82673e-8im      -3.56094e-16+1.34994e-15im
   6.12575e-7-3.58947e-7im      -1.15911e-15-1.97652e-15im
             ⋮               ⋱              ⋮
  3.56094e-16+1.34994e-15im       2.96285e-7-7.82673e-8im
  2.95304e-15+1.70884e-15im  …    6.32219e-8-1.09343e-7im
  1.82452e-15-4.92615e-16im      -1.29209e-

In [473]:
# second
# 2step version
function test2(ell, φ, θ, ψ, φ_pix, θ_pix)
    D_result = zeros(ComplexF64, 2ell+1, 2ell+1)
    #d = WignerD.wignerd(ell, pi/2)
    D_1st = WignerD.wignerD(ell, φ_pix, θ_pix, 0.0)
    for i in eachindex(φ)
        dφ = φ[i] - φ_pix
        dθ = θ[i] - θ_pix
        err, (α, β, γ) = check_split(φ_pix, θ_pix, dφ, dθ, ψ[i])
        D_2nd = WignerD.wignerD(ell, α, β, γ)
        D_result .+=  D_1st * D_2nd
    end
    return D_result./length(φ)
end

test2 (generic function with 1 method)

In [474]:
D2 = test2(ell, φ.+dφ, θ.+dθ, ψ, φ, θ)

201×201 Matrix{ComplexF64}:
 -1.49588e-13+7.66665e-14im  …   1.43688e-15+1.75632e-15im
  4.29012e-13-1.44733e-12im      1.97224e-15-1.04635e-15im
  4.23096e-12+7.72428e-12im      1.48885e-15+6.38068e-17im
 -4.01569e-11-1.16109e-11im     -1.07833e-15-7.97537e-17im
  1.49729e-10-8.31334e-11im     -1.18456e-15-6.10342e-16im
 -1.70381e-10+6.0235e-10im   …  -9.78828e-17-1.45658e-15im
  -1.02183e-9-1.81554e-9im       8.70183e-16+4.56877e-15im
   6.17105e-9+1.70698e-9im      -3.74237e-15-1.75013e-15im
  -1.59469e-8+9.0975e-9im        6.84447e-15+2.0365e-16im
   1.29209e-8-4.77887e-8im      -4.70609e-15+2.32849e-15im
   6.32219e-8+1.09343e-7im   …   2.77404e-15-4.18328e-15im
  -2.96285e-7-7.82673e-8im       4.01177e-16+3.66802e-15im
   6.12575e-7-3.58947e-7im      -1.22456e-15-3.04197e-15im
             ⋮               ⋱              ⋮
 -4.01177e-16+3.66802e-15im       2.96285e-7-7.82673e-8im
  2.77404e-15+4.18328e-15im  …    6.32219e-8-1.09343e-7im
  4.70609e-15+2.32849e-15im      -1.29209e-8

In [475]:
maximum(abs.(D1 .- D2))

1.9630007184965608e-13

In [476]:
function effective_wignerD(ell, φ, θ, ψ, φ_pix, θ_pix)
    D_result = zeros(ComplexF64, 2ell+1, 2ell+1)
    #d = WignerD.wignerd(ell, pi/2)
    D_1st = WignerD.wignerD(ell, φ_pix, θ_pix, 0.0)
    for i in eachindex(φ)
        dφ = φ[i] - φ_pix
        dθ = θ[i] - θ_pix
        err, (α, β, γ) = check_split(φ_pix, θ_pix, dφ, dθ, ψ[i])
        D_2nd = WignerD.wignerD(ell, α, β, γ)
        D_result .+=  D_1st * D_2nd
    end
    return D_result./length(φ)
end

effective_wignerD (generic function with 1 method)

In [477]:
D_effective = effective_wignerD(ell, φ.+dφ, θ.+dθ, ψ, φ, θ);

In [478]:
n_beam = sum(2*min(l, cb.mmax) + 1 for l in cc.lstart:cc.lstop)
D_beam = spzeros(n_beam, n_beam)

474×474 SparseMatrixCSC{Float64, Int64} with 0 stored entries:
⎡⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎦

In [479]:
function effective_wignerD_dir(ell, φ, θ, ψ, φ_pix, θ_pix)
    D_result = zeros(ComplexF64, 2ell+1, 2ell+1)
    #d = WignerD.wignerd(ell, pi/2)
    D_1st = WignerD.wignerD(ell, φ_pix, θ_pix, 0.0)
    for i in eachindex(φ)
        dφ = φ[i] - φ_pix
        dθ = θ[i] - θ_pix
        err, (α, β, γ) = check_split(φ_pix, θ_pix, dφ, dθ, ψ[i])
        D_2nd = WignerD.wignerD(ell, α, β, γ)
        D_result .+=  D_1st * D_2nd
    end
    return D_result./length(φ)
end

effective_wignerD_dir (generic function with 1 method)

In [480]:
d = WignerD.wignerd(ell, pi/2);

In [481]:
Dp = @time WignerD_calculator_fast(d, ell, φ, θ, ψ[1]);

  0.098838 seconds (23.71 k allocations: 2.905 MiB, 90.38% compilation time)


In [482]:
ellWignerD_calculator!

LoadError: UndefVarError: `ellWignerD_calculator!` not defined

In [483]:
# 複数角度サンプルの単純平均:
#   D_eff = (1/N) * Σ_i D(phi[i], theta[i], psi[i])
function WignerD_calculator!(result::AbstractMatrix{ComplexF64},
                             d, ell::Int,
                             phi::AbstractVector, theta::AbstractVector, psi::AbstractVector;
                             tmpD=nothing,
                             tmpB=nothing, w=nothing, p=nothing, q=nothing)
    N = length(phi)
    @assert length(theta) == N && length(psi) == N

    L = 2*ell + 1
    @assert size(result,1) == L && size(result,2) == L
    tmpD === nothing && (tmpD = Matrix{ComplexF64}(undef, L, L))

    fill!(result, 0.0 + 0.0im)
    @inbounds for i in eachindex(phi)
        WignerD_calculator!(tmpD, d, ell, phi[i], theta[i], psi[i];
                            tmpB=tmpB, w=w, p=p, q=q)
        result .+= tmpD
    end
    result ./= N
    return result
end

# 2step 平均（effective_wignerD 相当）:
#   D_eff = (1/N) * Σ_i D(phi_pix, theta_pix, 0) * D(α_i, β_i, γ_i)
#   where (α_i, β_i, γ_i) = check_split(phi_pix, theta_pix, dphi_i, dtheta_i, psi_i)
function WignerD_calculator!(result::AbstractMatrix{ComplexF64},
                             d, ell::Int,
                             phi::AbstractVector, theta::AbstractVector, psi::AbstractVector,
                             phi_pix, theta_pix;
                             eps=1e-12,
                             D1=nothing, D2=nothing, tmpMul=nothing,
                             tmpB1=nothing, w1=nothing, p1=nothing, q1=nothing,
                             tmpB2=nothing, w2=nothing, p2=nothing, q2=nothing)
    N = length(phi)
    @assert length(theta) == N && length(psi) == N

    L = 2*ell + 1
    @assert size(result,1) == L && size(result,2) == L
    D1     === nothing && (D1     = Matrix{ComplexF64}(undef, L, L))
    D2     === nothing && (D2     = Matrix{ComplexF64}(undef, L, L))
    tmpMul === nothing && (tmpMul = Matrix{ComplexF64}(undef, L, L))

    # 1st step: pixel center fixed rotation
    WignerD_calculator!(D1, d, ell, phi_pix, theta_pix, 0.0;
                        tmpB=tmpB1, w=w1, p=p1, q=q1)

    fill!(result, 0.0 + 0.0im)
    @inbounds for i in eachindex(phi)
        dphi = phi[i] - phi_pix
        dtheta = theta[i] - theta_pix
        _, (α, β, γ) = check_split(phi_pix, theta_pix, dphi, dtheta, psi[i]; eps=eps)

        WignerD_calculator!(D2, d, ell, α, β, γ;
                            tmpB=tmpB2, w=w2, p=p2, q=q2)
        mul!(tmpMul, D1, D2)
        result .+= tmpMul
    end

    result ./= N
    return result
end

WignerD_calculator! (generic function with 3 methods)

In [484]:
L = 2*ell + 1
result = zeros(ComplexF64, L, L)

# 単純平均
A = WignerD_calculator!(result, d, ell, φ.+dφ, θ.+dθ, ψ)

result = zeros(ComplexF64, L, L)
# 2step平均
B = WignerD_calculator!(result, d, ell, φ.+dφ, θ.+dθ, ψ, φ, θ)

201×201 Matrix{ComplexF64}:
 -1.61568e-13+8.70879e-14im  …  -4.17744e-17+9.49414e-18im
   4.2905e-13-1.4517e-12im       2.40839e-16-1.3738e-17im
   4.2337e-12+7.73085e-12im     -7.79973e-17+3.12059e-16im
 -4.01576e-11-1.16112e-11im     -6.23529e-17-1.79029e-16im
  1.49732e-10-8.31328e-11im      3.90825e-16+3.23569e-16im
 -1.70382e-10+6.02347e-10im  …  -2.24739e-16+9.3996e-17im
  -1.02183e-9-1.81553e-9im         2.581e-16-4.30969e-16im
   6.17105e-9+1.70697e-9im       8.64458e-17+3.18921e-16im
  -1.59469e-8+9.0975e-9im       -5.74889e-16-3.15297e-16im
   1.29209e-8-4.77887e-8im       5.63551e-16+7.39387e-17im
   6.32219e-8+1.09343e-7im   …  -3.66549e-16+3.99006e-16im
  -2.96285e-7-7.82673e-8im       2.62122e-16-4.27056e-16im
   6.12575e-7-3.58947e-7im       1.30913e-16+2.71171e-16im
             ⋮               ⋱              ⋮
 -2.60009e-16-4.30588e-16im       2.96285e-7-7.82673e-8im
 -3.66377e-16-3.96785e-16im  …    6.32219e-8-1.09343e-7im
 -5.63205e-16+7.20126e-17im      -1.29209e-8-

In [485]:
maximum(abs.(A.-B))

1.7828480455535504e-13

In [486]:
function effective_wignerD(ell, φ, θ, ψ, φ_pix, θ_pix)
    D_result = zeros(ComplexF64, 2ell+1, 2ell+1)
    #d = WignerD.wignerd(ell, pi/2)
    D_1st = WignerD.wignerD(ell, φ_pix, θ_pix, 0.0)
    for i in eachindex(φ)
        dφ = φ[i] - φ_pix
        dθ = θ[i] - θ_pix
        err, (α, β, γ) = check_split(φ_pix, θ_pix, dφ, dθ, ψ[i])
        D_2nd = WignerD.wignerD(ell, α, β, γ)
        D_result .+=  D_1st * D_2nd
    end
    return D_result./length(φ)
end

effective_wignerD (generic function with 1 method)

In [487]:
d_beam = spzeros(ComplexF64,n_beam, n_beam)


474×474 SparseMatrixCSC{ComplexF64, Int64} with 0 stored entries:
⎡⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎦

In [488]:
D_beam = spzeros(ComplexF64,n_beam, n_beam)

474×474 SparseMatrixCSC{ComplexF64, Int64} with 0 stored entries:
⎡⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎦

In [489]:
function D_2ndtest(cb, cc, α, β, γ)
    n_beam = sum(2*min(l, cb.mmax) + 1 for l in cc.lstart:cc.lstop)
    n_sky = sum(2*l + 1 for l in cc.lstart:cc.lstop)
    D_beam = spzeros(ComplexF64, n_sky, n_beam)
    for l in cc.lstart:cc.lstop
        #@show l
        #d_temp = WignerD.wignerd(ell,pi/2)
        n_ = min(l, cb.mmax)
        @inbounds for m in -l:l
            m_idx = lmr_idx(l=l, m=m, lstart=cc.lstart, mmax=cc.lstop)
            @show m_idx
            @inbounds for n in -n_:n_
                n_idx = lmr_idx(l=l, m=n, lstart=cc.lstart, mmax=cb.mmax)
                D_beam[m_idx, n_idx] = WignerD.wignerDjmn(l, m, n, α, β, γ)
            end
        end
    end
    return D_beam
end

D_2ndtest (generic function with 1 method)

In [490]:
function D_1sttest(cb, cc, φ, θ, ψ)
    n_sky = sum(2*l + 1 for l in cc.lstart:cc.lstop)
    D_beam = spzeros(ComplexF64, n_sky, n_sky)
    for l in cc.lstart:cc.lstop
        #@show l
        #d_temp = WignerD.wignerd(ell,pi/2)
        m_ = l
        @inbounds for m in -m_:m_
            m_idx = lmr_idx(l=l, m=m, lstart=cc.lstart, mmax=cc.lstop)
            @show m_idx
            @inbounds for n in -m_:m_
                n_idx = lmr_idx(l=l, m=n, lstart=cc.lstart, mmax=cc.lstop)
                D_beam[m_idx, n_idx] = WignerD.wignerDjmn(l, m, n, φ, θ, ψ)
            end
        end
    end
    return D_beam
end

D_1sttest (generic function with 1 method)

In [491]:
nptg = 1
θ = pi/3
φ = pi/4
dθ = rand()*1e-2
dφ = rand()*1e-2
ψ = rand()*2pi

5.132609304028245

In [492]:
cb.mmax = 10
n_beam = sum(2*min(l, cb.mmax) + 1 for l in cc.lstart:cc.lstop)


1906

In [493]:
err_, (alphas_, betas_, gammas_) = check_split(φ, θ, dφ, dθ, ψ)

(8.430756093247282e-15, (0.9029300098738107, 0.008505227286915927, -2.0496797981538))

In [494]:
b = D_2ndtest(cb, cc, alphas_, betas_, gammas_)

m_idx = 1
m_idx = 2
m_idx = 3
m_idx = 4
m_idx = 5
m_idx = 6
m_idx = 7
m_idx = 8
m_idx = 9
m_idx = 10
m_idx = 11
m_idx = 12
m_idx = 13
m_idx = 14
m_idx = 15
m_idx = 16
m_idx = 17
m_idx = 18
m_idx = 19
m_idx = 20
m_idx = 21
m_idx = 22
m_idx = 23
m_idx = 24
m_idx = 25
m_idx = 26
m_idx = 27
m_idx = 28
m_idx = 29
m_idx = 30
m_idx = 31
m_idx = 32
m_idx = 33
m_idx = 34
m_idx = 35
m_idx = 36
m_idx = 37
m_idx = 38
m_idx = 39
m_idx = 40
m_idx = 41
m_idx = 42
m_idx = 43
m_idx = 44
m_idx = 45
m_idx = 46
m_idx = 47
m_idx = 48
m_idx = 49
m_idx = 50
m_idx = 51
m_idx = 52
m_idx = 53
m_idx = 54
m_idx = 55
m_idx = 56
m_idx = 57
m_idx = 58
m_idx = 59
m_idx = 60
m_idx = 61
m_idx = 62
m_idx = 63
m_idx = 64
m_idx = 65
m_idx = 66
m_idx = 67
m_idx = 68
m_idx = 69
m_idx = 70
m_idx = 71
m_idx = 72
m_idx = 73
m_idx = 74
m_idx = 75
m_idx = 76
m_idx = 77
m_idx = 78
m_idx = 79
m_idx = 80
m_idx = 81
m_idx = 82
m_idx = 83
m_idx = 84
m_idx = 85
m_idx = 86
m_idx = 87
m_idx = 88
m_idx = 89
m_idx = 90
m_idx = 91
m_idx = 

9216×1906 SparseMatrixCSC{ComplexF64, Int64} with 192763 stored entries:
⎡⠻⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠘⣆⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⢹⡄⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⢷⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠸⡆⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⢷⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⢸⡄⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⣧⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⢸⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠸⡇⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⣧⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⢸⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠸⡆⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⡇⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⢿⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⢸⡀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠘⡇⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⡇⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⢻⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⢸⡀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠘⡇⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⡇⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⣷⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⡆⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡇⎦

In [495]:
a = D_1sttest(cb, cc, φ, θ, 0)

m_idx = 1
m_idx = 2
m_idx = 3
m_idx = 4
m_idx = 5
m_idx = 6
m_idx = 7
m_idx = 8
m_idx = 9
m_idx = 10
m_idx = 11
m_idx = 12
m_idx = 13
m_idx = 14
m_idx = 15
m_idx = 16
m_idx = 17
m_idx = 18
m_idx = 19
m_idx = 20
m_idx = 21
m_idx = 22
m_idx = 23
m_idx = 24
m_idx = 25
m_idx = 26
m_idx = 27
m_idx = 28
m_idx = 29
m_idx = 30
m_idx = 31
m_idx = 32
m_idx = 33
m_idx = 34
m_idx = 35
m_idx = 36
m_idx = 37
m_idx = 38
m_idx = 39
m_idx = 40
m_idx = 41
m_idx = 42
m_idx = 43
m_idx = 44
m_idx = 45
m_idx = 46
m_idx = 47
m_idx = 48
m_idx = 49
m_idx = 50
m_idx = 51
m_idx = 52
m_idx = 53
m_idx = 54
m_idx = 55
m_idx = 56
m_idx = 57
m_idx = 58
m_idx = 59
m_idx = 60
m_idx = 61
m_idx = 62
m_idx = 63
m_idx = 64
m_idx = 65
m_idx = 66
m_idx = 67
m_idx = 68
m_idx = 69
m_idx = 70
m_idx = 71
m_idx = 72
m_idx = 73
m_idx = 74
m_idx = 75
m_idx = 76
m_idx = 77
m_idx = 78
m_idx = 79
m_idx = 80
m_idx = 81
m_idx = 82
m_idx = 83
m_idx = 84
m_idx = 85
m_idx = 86
m_idx = 87
m_idx = 88
m_idx = 89
m_idx = 90
m_idx = 91
m_idx = 

9216×9216 SparseMatrixCSC{ComplexF64, Int64} with 1179616 stored entries:
⎡⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠈⠻⢆⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠿⣧⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠿⣧⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠛⣤⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠿⣧⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠿⣧⣀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⠻⣦⡄⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⢻⣶⣀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⠻⣦⡄⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⢻⣶⎦

In [439]:
c=a*b

147456×7954 SparseMatrixCSC{ComplexF64, Int64} with 194488 stored entries:
⎡⢧⠀⠀⎤
⎢⠸⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎢⠀⠀⠀⎥
⎣⠀⠀⠀⎦

In [440]:
d = D_2ndtest(cb, cc, φ+dφ, θ+dθ, ψ)

m_idx = 1
m_idx = 2
m_idx = 3
m_idx = 4
m_idx = 5
m_idx = 6
m_idx = 7
m_idx = 8
m_idx = 9
m_idx = 10
m_idx = 11
m_idx = 12
m_idx = 13
m_idx = 14
m_idx = 15
m_idx = 16
m_idx = 17
m_idx = 18
m_idx = 19
m_idx = 20
m_idx = 21
m_idx = 22
m_idx = 23
m_idx = 24
m_idx = 25
m_idx = 26
m_idx = 27
m_idx = 28
m_idx = 29
m_idx = 30
m_idx = 31
m_idx = 32
m_idx = 33
m_idx = 34
m_idx = 35
m_idx = 36
m_idx = 37
m_idx = 38
m_idx = 39
m_idx = 40
m_idx = 41
m_idx = 42
m_idx = 43
m_idx = 44
m_idx = 45
m_idx = 46
m_idx = 47
m_idx = 48
m_idx = 49
m_idx = 50
m_idx = 51
m_idx = 52
m_idx = 53
m_idx = 54
m_idx = 55
m_idx = 56
m_idx = 57
m_idx = 58
m_idx = 59
m_idx = 60
m_idx = 61
m_idx = 62
m_idx = 63
m_idx = 64
m_idx = 65
m_idx = 66
m_idx = 67
m_idx = 68
m_idx = 69
m_idx = 70
m_idx = 71
m_idx = 72
m_idx = 73
m_idx = 74
m_idx = 75
m_idx = 76
m_idx = 77
m_idx = 78
m_idx = 79
m_idx = 80
m_idx = 81
m_idx = 82
m_idx = 83
m_idx = 84
m_idx = 85
m_idx = 86
m_idx = 87
m_idx = 88
m_idx = 89
m_idx = 90
m_idx = 91
m_idx = 

Excessive output truncated after 524295 bytes.

m_idx = 37943
m_idx = 37944
m_idx = 37945
m_idx = 37946
m_idx = 37947
m_idx = 37948
m_idx = 37949
m_idx = 37950
m_idx = 37951
m_idx = 37952
m_idx = 37953
m_idx = 37954
m_idx = 37955
m_idx = 37956
m_idx = 37957
m_idx = 37958
m_idx = 37959
m_idx = 37960
m_idx = 37961
m_idx = 37962
m_idx = 37963
m_idx = 37964
m_idx = 37965
m_idx = 37966
m_idx = 37967
m_idx = 37968
m_idx = 37969
m_idx = 37970
m_idx = 37971
m_idx = 37972
m_idx = 37973
m_idx = 37974
m_idx = 37975
m_idx = 37976
m_idx = 37977
m_idx = 37978
m_idx = 37979
m_idx = 37980
m_idx = 37981
m_idx = 37982
m_idx = 37983
m_idx = 37984
m_idx = 37985
m_idx = 37986
m_idx = 37987
m_idx = 37988
m_idx = 37989
m_idx = 37990
m_idx = 37991
m_idx = 37992
m_idx = 37993
m_idx = 37994
m_idx = 37995
m_idx = 37996
m_idx = 37997
m_idx = 37998
m_idx = 37999
m_idx = 38000
m_idx = 38001
m_idx = 38002
m_idx = 38003
m_idx = 38004
m_idx = 38005
m_idx = 38006
m_idx = 38007
m_idx = 38008
m_idx = 38009
m_idx = 38010
m_idx = 38011
m_idx = 38012
m_idx = 38013
m_idx 

147456×7954 SparseMatrixCSC{ComplexF64, Int64} with 3095806 stored entries:
⎡⢧⠀⠀⎤
⎢⢸⠀⠀⎥
⎢⢸⠀⠀⎥
⎢⢸⡀⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⢳⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⡀⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎣⠀⠀⡇⎦

In [441]:
maximum(abs.(d-c))

0.6459050901836227

In [ ]:
c[6:15,6:15]

10×10 SparseMatrixCSC{ComplexF64, Int64} with 52 stored entries:
 0.000611987+0.00479749im  …            ⋅    
    0.419343+0.320889im                 ⋅    
   -0.496866+0.0687921im                ⋅    
    0.130844-0.174833im                 ⋅    
             ⋅                0.0325515+0.0588982im
             ⋅             …    0.21266+0.0600375im
             ⋅                 0.374034-0.211974im
             ⋅                 0.133924-0.49455im
             ⋅                -0.126469-0.217694im
             ⋅                 0.276327+0.0716648im

In [ ]:
d[6:15,6:15]

10×10 SparseMatrixCSC{ComplexF64, Int64} with 52 stored entries:
 0.000611987+0.00479749im     0.371373+0.375368im  …            ⋅    
    0.419343+0.320889im      -0.129831+0.0im                    ⋅    
   -0.496866+0.0687921im     -0.371373+0.375368im               ⋅    
    0.130844-0.174833im    -0.00493486-0.461225im               ⋅    
             ⋅                         ⋅              0.0325515+0.0588982im
             ⋅                         ⋅           …    0.21266+0.0600375im
             ⋅                         ⋅               0.374034-0.211974im
             ⋅                         ⋅               0.133924-0.49455im
             ⋅                         ⋅              -0.126469-0.217694im
             ⋅                         ⋅               0.276327+0.0716648im

In [376]:
WignerD.wignerDjmn(4, 1, 1, pi/3, pi/2, pi/6)

2.2962127484012914e-17 - 0.37500000000000067im

In [369]:
fieldnames(typeof(WignerD.wignerdjmn))

()

1914×1914 SparseMatrixCSC{Float64, Int64} with 0 stored entries:
⎡⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎦

In [159]:
fieldnames(typeof(cb))

(:lmax, :mmax, :blm)

In [157]:
cc.mmax_calculate

2

In [134]:
D_beam[1,1:4] .=+ Vector(1:1:4).*0.0

4-element view(::SparseMatrixCSC{Float64, Int64}, 1, 1:4) with eltype Float64:
 0.0
 0.0
 0.0
 0.0

In [135]:
D_beam[1,1:4]

4-element SparseVector{Float64, Int64} with 0 stored entries

In [153]:
lmr_idx(l=2, m=0, lstart=0, mmax=2)

7

In [58]:
WignerD_calculator!(result::AbstractMatrix{ComplexF64},
                             d, ell::Int, phi, theta, psi;
                             tmpB=nothing, w=nothing, p=nothing, q=nothing)

LoadError: UndefVarError: `result` not defined

In [33]:
cc.lstop

383

In [23]:
D

1000×1000 SparseMatrixCSC{Float64, Int64} with 0 stored entries:
⎡⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎦

In [52]:
function lmr_idx(;l::Int, m::Int, lstart::Int)
    l >= lstart || throw(ArgumentError("l must be >= lstart"))
    (-l <= m <= l) || throw(ArgumentError("m must be in [-l, l]"))
    offset = sum(2k + 1 for k in lstart:l-1; init=0)
    idx = offset + (m + l) + 1   # m=-l -> +1, m=l -> +2l+1
    return idx
end

lmr_idx (generic function with 1 method)

In [53]:
lmr_idx(l=2, m=0, lstart=0)

7

In [55]:
lmr_idx

lmr_idx (generic function with 1 method)

In [30]:
lmr_idx(l=2, m=0, lstart=cc.lstart, mmax=cc.mmax_calculate)

7

In [37]:
lmr_idx(l=3, m=2, lstart=cc.lstart, mmax=cc.mmax_calculate)

14

In [32]:
lmr_idx(l=2, m=-2, lstart=cc.lstart, mmax=cc.mmax_calculate)

5

In [33]:
lmr_idx(l=2, m=-1, lstart=cc.lstart, mmax=cc.mmax_calculate)

6

In [25]:
n = sum(2*min(l, 3) + 1 for l in cc.lstart:cc.lstop)

2676

In [26]:
1+3+5+7+7*(cc.lstop-3)

2676

In [ ]:
n = sum(2*min(l, cb.mmax) + 1 for l in cc.lstart:cc.lstop)


147456

In [99]:
alm_idx(l=4, m=3, lmax=4, mmax=3)

14

In [70]:
m = -2
phase = isodd(m) ? -1.0 : 1.0


1.0

In [ ]:
idx_start, idx_stop = lrange_idx(3, 2)

(6, 12)

In [58]:
lmr_idx(2, -2, 0)

5

In [46]:
maximum(cc.l_calculate)

383

In [ ]:
function lrange_idx(l::Int, lstart::Int)
    l >= lstart || throw(ArgumentError("l must be >= lstart"))
    offset = sum(2k + 1 for k in lstart:l-1)
    start = offset + 1
    stop  = start + (2l + 1) - 1
    return start, stop
end


In [43]:
for l in cc.l_calculate
    println("l = $l")
end

l = 0
l = 1
l = 2
l = 3
l = 4
l = 5
l = 6
l = 7
l = 8
l = 9
l = 10
l = 11
l = 12
l = 13
l = 14
l = 15
l = 16
l = 17
l = 18
l = 19
l = 20
l = 21
l = 22
l = 23
l = 24
l = 25
l = 26
l = 27
l = 28
l = 29
l = 30
l = 31
l = 32
l = 33
l = 34
l = 35
l = 36
l = 37
l = 38
l = 39
l = 40
l = 41
l = 42
l = 43
l = 44
l = 45
l = 46
l = 47
l = 48
l = 49
l = 50
l = 51
l = 52
l = 53
l = 54
l = 55
l = 56
l = 57
l = 58
l = 59
l = 60
l = 61
l = 62
l = 63
l = 64
l = 65
l = 66
l = 67
l = 68
l = 69
l = 70
l = 71
l = 72
l = 73
l = 74
l = 75
l = 76
l = 77
l = 78
l = 79
l = 80
l = 81
l = 82
l = 83
l = 84
l = 85
l = 86
l = 87
l = 88
l = 89
l = 90
l = 91
l = 92
l = 93
l = 94
l = 95
l = 96
l = 97
l = 98
l = 99
l = 100
l = 101
l = 102
l = 103
l = 104
l = 105
l = 106
l = 107
l = 108
l = 109
l = 110
l = 111
l = 112
l = 113
l = 114
l = 115
l = 116
l = 117
l = 118
l = 119
l = 120
l = 121
l = 122
l = 123
l = 124
l = 125
l = 126
l = 127
l = 128
l = 129
l = 130
l = 131
l = 132
l = 133
l = 134
l = 135
l = 136
l = 137
l = 13

In [62]:
hcat(blm_harmonic.blmT.alm,
     blm_harmonic.blmE.alm,
     blm_harmonic.blmB.alm) 

73920×3 Matrix{ComplexF64}:
 0.282095+0.0im  0.0+0.0im  0.0+0.0im
 0.488576+0.0im  0.0+0.0im  0.0+0.0im
 0.630679+0.0im  0.0+0.0im  0.0+0.0im
 0.746107+0.0im  0.0+0.0im  0.0+0.0im
  0.84582+0.0im  0.0+0.0im  0.0+0.0im
 0.934832+0.0im  0.0+0.0im  0.0+0.0im
  1.01593+0.0im  0.0+0.0im  0.0+0.0im
  1.09087+0.0im  0.0+0.0im  0.0+0.0im
  1.16081+0.0im  0.0+0.0im  0.0+0.0im
  1.22659+0.0im  0.0+0.0im  0.0+0.0im
  1.28882+0.0im  0.0+0.0im  0.0+0.0im
  1.34798+0.0im  0.0+0.0im  0.0+0.0im
  1.40444+0.0im  0.0+0.0im  0.0+0.0im
         ⋮                  
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0

In [59]:
[blm_harmonic.blmT.alm, blm_harmonic.blmE.alm, blm_harmonic.blmB.alm]

3-element Vector{Vector{ComplexF64}}:
 [0.28209479177387814 + 0.0im, 0.4885756718694027 + 0.0im, 0.6306791852123736 + 0.0im, 0.7461067059896269 + 0.0im, 0.8458196072043231 + 0.0im, 0.9348319547260208 + 0.0im, 1.0159345688951988 + 0.0im, 1.0908692242925033 + 0.0im, 1.1608087185364095 + 0.0im, 1.226586793157029 + 0.0im  …  0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im]
 [0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im  …  0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im]
 [0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im  …  0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im]

In [35]:
blm_harmonic.blmT.alm

73920-element Vector{ComplexF64}:
 0.28209479177387814 + 0.0im
  0.4885756718694027 + 0.0im
  0.6306791852123736 + 0.0im
  0.7461067059896269 + 0.0im
  0.8458196072043231 + 0.0im
  0.9348319547260208 + 0.0im
  1.0159345688951988 + 0.0im
  1.0908692242925033 + 0.0im
  1.1608087185364095 + 0.0im
   1.226586793157029 + 0.0im
   1.288820860640887 + 0.0im
  1.3479829400254342 + 0.0im
  1.4044432431790195 + 0.0im
                     ⋮
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im

In [47]:
mmax = lmax
nalm = Healpix.numberOfAlms(lmax, mmax)
blmT = Healpix.Alm(lmax, mmax, zeros(ComplexF64, nalm))

Alm{ComplexF64, Vector{ComplexF64}}(ComplexF64[0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im  …  0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im], 383, 383, 767)

In [52]:
blmT

Alm{ComplexF64, Vector{ComplexF64}}(ComplexF64[0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im  …  0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im], 383, 383, 767)

In [53]:
typeof(blmT)

Alm{ComplexF64, Vector{ComplexF64}}

70000*70000*8

In [39]:
70000*70000

4900000000

In [4]:
function ConvolutionSky(;
    nside::Int = 128,
    lmax::Int = 3*nside - 1,
    alm::Matrix{ComplexF64} = fill(1.0 + 1.0im, 2, 2),
    realization::Int = 1
)
    return ConvolutionSky(nside, lmax, alm, realization)
end

ConvolutionSky

In [25]:
mutable struct ConvolutionSky{I<:Int, MC<:Matrix{Complex{Float64}}}
    nside::I
    lmax::I
    alm::MC
    realization::I
end

LoadError: invalid redefinition of constant Main.ConvolutionSky

In [40]:
nalm = Healpix.numberOfAlms(1000, 1000)


501501

In [5]:
nside = 16
cp = gen_ConvolutionParams_pc(nside = nside);
cp.beam_mmax = 10

10

In [6]:
lmax = nside*3-1
res = Resolution(nside)
beammmax = 10

10

In [7]:
# input beam
size = alm_idx(lmax,lmax,lmax)
blm_ = zeros(ComplexF64, 3, size)
blm = hp.blm_gauss(deg2rad(0.5), lmax = lmax, pol = true)
blm_[1,1:length(blm[1,:])] = blm[1,:]
blm_[2,1:length(blm[1,:])] = -blm[2,:]*sqrt(2)
blm_[3,1:length(blm[1,:])] = -blm[3,:]*sqrt(2);
@time blm_full = get_reorder_blm_optimized(blm_, lmax, beammmax);

LoadError: UndefVarError: `hp` not defined

In [8]:
alm = npzread("./inputs/alm=CMB=r0=nside$nside.npy")
@time alm_full = get_reorder_alm_optimized(alm, lmax);

  0.233391 seconds (245.43 k allocations: 16.078 MiB, 3.35% gc time, 98.94% compilation time)


In [9]:
data_path = "/data/n/n339/wigner_file/"

"/data/n/n339/wigner_file/"

In [10]:
@time initial_wignerd = [zeros(ComplexF64,2*i+1,2*i+1) for i in 0:cp.lmax]
@time initial_wignerd_ = [zeros(ComplexF64,2*i+1,2*i+1) for i in 0:cp.lmax]
@time eff_wignerD = [zeros(ComplexF64,2*i+1,2*i+1) for i in 0:cp.lmax];
@time eff_wignerD_ = [zeros(ComplexF64,2*i+1,2*i+1) for i in 0:cp.lmax];

  0.071945 seconds (29.82 k allocations: 4.259 MiB, 40.88% gc time, 57.19% compilation time)
  0.029705 seconds (21.82 k allocations: 3.724 MiB, 97.19% compilation time)
  0.028619 seconds (21.82 k allocations: 3.724 MiB, 96.99% compilation time)
  0.036557 seconds (21.82 k allocations: 3.724 MiB, 96.14% compilation time)


In [11]:
initial_wignerd[2]

3×3 Matrix{ComplexF64}:
 0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im

In [12]:
function initialwignerds_array_(cp, theta, path, initial_wignerd)
    count = 1
    for l in cp.l_range[1]:cp.l_range[2]
        initial_wignerd[count] = swignerd_calc(l, theta, path)
        count += 1
    end
   return initial_wignerd
end

initialwignerds_array_ (generic function with 1 method)

In [13]:
function effective_wignerD_onz_(cp, ψs, initial_wignerd_onz_)
    for m in -cp.lmax-2:cp.lmax+2
        initial_wignerd_onz_[cp.lmax+1+m] = mean(exp.(-1im*ψs*m))
    end
    return initial_wignerd_onz_
end

effective_wignerD_onz_ (generic function with 1 method)

In [14]:
npix = nside2npix(nside)
utv = unique_theta_val(cp);

In [15]:
initial_wignerd_onz = zeros(ComplexF64, 3, 2*cp.lmax+1)
conv_binned_map = zeros(3,npix)
result_d = zeros(ComplexF64,3)
@time for i in 1:length(utv)
    @show i
    start_pix, stop_pix = unique_theta_num(i, cp)
    initialwignerds_array(cp, utv[i], data_path, initial_wignerd);
    for pix_num in start_pix:stop_pix
        center_th, center_phi = pix2angRing(res, pix_num)
        calc_psi = rand(100)*2pi
        effective_wignerD_onz(cp, calc_psi[:], initial_wignerd_onz)
        for j in 1:3
            get_pc_total_effective_wignerD(cp, center_phi, initial_wignerd_onz[j,:], initial_wignerd, eff_wignerD);
            result_d[j] = eff_convolver_optimized(alm_full, blm_full, eff_wignerD, beammmax)
        end
        conv = binned_mapmaker(calc_psi, result_d)
        conv_binned_map[:,pix_num] .= conv
    end
end

i = 1


LoadError: SystemError: opening file "/data/n/n339/wigner_file/dmatrices=ell0.npy": No such file or directory

In [ ]:
alm_smooth = hp.smoothalm([alm[1,:],alm[2,:],alm[3,:]],fwhm = deg2rad(0.5))
in_map = hp.alm2map([alm_smooth[1],alm_smooth[2],alm_smooth[3]], nside = nside)

LoadError: UndefVarError: `alm` not defined